# Notebook 02 — Feature Engineering
**Project:** Credit Card Customer Segmentation  
**Goal:** Clean the data, impute missing values, engineer behavioural features, and scale for clustering.

---
### Flow
1. Load raw data
2. Handle missing values
3. Engineer new behavioural features
4. Outlier analysis
5. Scale features (StandardScaler)
6. Validate final feature matrix
7. Save processed data

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

from src.data_loader import load_raw_data
df = load_raw_data()
print('Raw shape:', df.shape)
df.head()

## 1. Handle Missing Values

In [ ]:
df = df.drop(columns=['CUST_ID'])

print('Missing before imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])

# Impute with median — robust to the heavy right skew in monetary features
for col in df.columns:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f'  Imputed {col} with median = {median_val:.2f}')

print(f'\nMissing after imputation: {df.isnull().sum().sum()}')
print(f'Shape: {df.shape}')

## 2. Feature Engineering

Based on EDA findings, we create 4 new behavioural ratio features:

| Feature | Formula | Business Meaning |
|---|---|---|
| `PURCHASES_TO_LIMIT_RATIO` | PURCHASES / CREDIT_LIMIT | How much of limit is actually spent |
| `CASH_ADVANCE_RATIO` | CASH_ADVANCE / BALANCE | Reliance on cash advances vs balance |
| `PAYMENT_TO_MINIMUM_RATIO` | PAYMENTS / MINIMUM_PAYMENTS | How aggressively customers pay down debt |
| `MONTHLY_AVG_PURCHASE` | PURCHASES / TENURE | Average monthly spending rate |

In [ ]:
df['PURCHASES_TO_LIMIT_RATIO']  = (df['PURCHASES'] / (df['CREDIT_LIMIT'] + 1)).round(4)
df['CASH_ADVANCE_RATIO']        = (df['CASH_ADVANCE'] / (df['BALANCE'] + 1)).round(4)
df['PAYMENT_TO_MINIMUM_RATIO']  = (df['PAYMENTS'] / (df['MINIMUM_PAYMENTS'] + 1)).round(4)
df['MONTHLY_AVG_PURCHASE']      = (df['PURCHASES'] / df['TENURE']).round(2)

print('New engineered features:')
new_features = ['PURCHASES_TO_LIMIT_RATIO', 'CASH_ADVANCE_RATIO',
                'PAYMENT_TO_MINIMUM_RATIO', 'MONTHLY_AVG_PURCHASE']
print(df[new_features].describe().round(3))

In [ ]:
# Visualise engineered features
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
colors = ['steelblue', 'tomato', 'seagreen', 'orange']

for ax, col, color in zip(axes, new_features, colors):
    ax.hist(df[col], bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].median(), color='black', linestyle='--', linewidth=1.2,
               label=f'Median: {df[col].median():.2f}')
    ax.set_title(col.replace('_', ' ').title(), fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Engineered Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/06_engineered_features.png', bbox_inches='tight')
plt.show()

## 3. Outlier Analysis

In [ ]:
monetary_cols = ['BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'CREDIT_LIMIT', 'PAYMENTS']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, monetary_cols):
    ax.boxplot(df[col], patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6),
               medianprops=dict(color='black', linewidth=2))
    ax.set_title(col, fontsize=9)
    ax.set_ylabel('Value ($)')

plt.suptitle('Outlier Analysis — Monetary Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/07_outliers.png', bbox_inches='tight')
plt.show()

print('Decision: RETAIN outliers — they represent genuine high-spend/high-risk customers')
print('StandardScaler will reduce their relative influence during clustering.')

## 4. Feature Scaling (StandardScaler)

KMeans is distance-based — features on different scales dominate the distance calculation.
We use `StandardScaler` (mean=0, std=1) to give every feature equal weight.

In [ ]:
scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df),
    columns=df.columns
)

print('Before scaling — BALANCE stats:')
print(f'  Mean={df["BALANCE"].mean():.2f}, Std={df["BALANCE"].std():.2f}')
print('After scaling — BALANCE stats:')
print(f'  Mean={df_scaled["BALANCE"].mean():.4f}, Std={df_scaled["BALANCE"].std():.4f}')
print(f'\nFinal feature matrix: {df_scaled.shape[0]:,} rows x {df_scaled.shape[1]} features')

## 5. Final Feature Matrix

In [ ]:
print(f'Total features for clustering: {df_scaled.shape[1]}')
print('\nFeature list:')
for i, col in enumerate(df_scaled.columns, 1):
    print(f'  {i:2}. {col}')

df_scaled.head()

In [ ]:
# Save via pipeline
from src.preprocessing import preprocess
from src.data_loader import load_raw_data
df_scaled_out, df_unscaled_out = preprocess(load_raw_data(), save=True)
print('Processed data saved to data/processed/')

## Summary

| Step | Action | Result |
|---|---|---|
| Drop ID | Removed CUST_ID | 17 features remain |
| Imputation | Median fill | 0 missing values |
| Feature Engineering | 4 ratio features | 21 total features |
| Outlier Decision | Retained | Genuine high-value segments |
| Scaling | StandardScaler | Mean=0, Std=1 per feature |

> **Next step:** `03_Model_Clustering.ipynb` — determine optimal K using Elbow & Silhouette, fit KMeans, and interpret customer segments.